# Cluster-Based CV Folds

Create cluster-held-out cross-validation folds from a training parquet file, using the same method as
`cv_method: 'cluster'` in `ml-pipeline-aircheck` (`_build_cluster_folds` in `model_utils.py`).
The output `.npz` works with every k-fold notebook here, in place of the stratified folds from
[stratified_cv.py](stratified_cv.py).

**Method**
1. Read only the fingerprint (ECFP4), label and SMILES columns; drop rows with a missing fingerprint or label.
2. Cluster **only the positives**: Tanimoto distance on the binarized fingerprints, agglomerative
   clustering into `N_CLUSTERS` groups with `LINKAGE` linkage.
3. Each fold's test positives come from whole clusters (largest unused cluster first, topped up with the
   smallest ones until there are `TARGET_TEST_POS_PER_FOLD` positives). **Every positive in those clusters is
   removed from that fold's training set**, so each fold is tested on chemistry it has not trained on.
4. Test negatives are a random `TEST_NEG_FRACTION` of all negatives, drawn fresh for each fold
   (so they can repeat across folds); the remaining negatives are used for training.
5. Up to `TARGET_N_FOLDS` folds are built, stopping early when unused clusters run out.

**Environment:** run with the `biomed-multiview` kernel.

**Output** (in `OUTPUT_DIR`, by default `aircheck/output/<DATASET>/cv_folds/`):
```
<train name>_cluster_cv_data.csv              # SMILES, LABEL, parquet_row: the rows the fold indices refer to
<train name>_cluster_cv_folds.npz             # fold0_train, fold0_test, fold1_train, ... (0-based rows of the CSV)
```
Use the CSV as `DATA_PATH` and the `.npz` as `CV_INDICES_PATH` in the k-fold notebooks, and extract
embeddings from the same CSV for the LightGBM notebooks.

## 1. Inputs and settings

**Edit these before running.**

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Training data (parquet) and its columns
TRAIN_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/train.parquet")
FINGERPRINT_COL = "ECFP4"
LABEL_COL = "LABEL"
SMILES_COL = "SMILES"

# Where to save the CSV and .npz (None = the train file's folder)
OUTPUT_DIR = Path("../output") / DATASET / "cv_folds"

# Use only the first N rows of the parquet (None = all rows), like nrows_train in the pipeline
NROWS = None

In [2]:
# Fold settings: the values hard-coded in ml-pipeline-aircheck, except TARGET_N_FOLDS (10 there)
N_CLUSTERS = 20                  # agglomerative clusters built from the positives
LINKAGE = "average"              # linkage on the Tanimoto distance matrix
TARGET_N_FOLDS = 5               # maximum number of folds (fewer if clusters run out)
TARGET_TEST_POS_PER_FOLD = 100   # positives in each fold's test set
TEST_NEG_FRACTION = 0.25         # fraction of all negatives sampled for each fold's test set
SEED = 42                        # picks test positives within the clusters and samples test negatives

In [3]:
TRAIN_PATH = Path(TRAIN_PATH).expanduser()
assert TRAIN_PATH.exists(), f"Train file not found: {TRAIN_PATH}"
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser() if OUTPUT_DIR is not None else TRAIN_PATH.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Train  : {TRAIN_PATH}")
print(f"Output : {OUTPUT_DIR}")

Train  : ~/repos/biomed-multi-view/aircheck/sample_data/train.parquet
Output : ../output/sample-data/cv_folds


## 2. Load the needed columns

Only the fingerprint, label and SMILES columns are read, so other fingerprint columns in a large file
are never loaded. Fingerprints are kept only for the positives, which are all that the clustering needs.

In [4]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

columns = pq.ParquetFile(TRAIN_PATH).schema_arrow.names
missing = [c for c in (FINGERPRINT_COL, LABEL_COL, SMILES_COL) if c not in columns]
assert not missing, f"{TRAIN_PATH.name} has no column(s) {missing}; available: {columns}"

df = pd.read_parquet(TRAIN_PATH, columns=[FINGERPRINT_COL, LABEL_COL, SMILES_COL])
if NROWS is not None:
    df = df.head(NROWS)
df["parquet_row"] = np.arange(len(df))
# Same rows as the pipeline: drop missing fingerprint or label, then renumber
df = df.dropna(subset=[FINGERPRINT_COL, LABEL_COL]).reset_index(drop=True)

y = df[LABEL_COL].astype(int).to_numpy()
assert set(np.unique(y)) <= {0, 1}, f"Labels must be 0/1, found {np.unique(y)}"
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]


def to_bits(value) -> np.ndarray:
    # Fingerprints are stored either as arrays or as comma-separated strings
    if isinstance(value, str):
        value = np.fromstring(value, sep=",", dtype=np.float32)
    return np.asarray(value) > 0


X_pos = np.vstack([to_bits(v) for v in df[FINGERPRINT_COL].iloc[pos_idx]]).astype(np.uint8)
df = df.drop(columns=[FINGERPRINT_COL])  # no longer needed

print(f"{len(df):,} rows: {len(pos_idx):,} positives, {len(neg_idx):,} negatives; "
      f"{FINGERPRINT_COL} has {X_pos.shape[1]} bits")
n_missing_smiles = int(df[SMILES_COL].isna().sum())
if n_missing_smiles:
    print(f"WARNING: {n_missing_smiles} rows have no SMILES; the MMELON notebooks will stop on these rows")
assert len(pos_idx) >= TARGET_TEST_POS_PER_FOLD, (
    f"Only {len(pos_idx)} positives; need at least TARGET_TEST_POS_PER_FOLD = {TARGET_TEST_POS_PER_FOLD}"
)
print(f"Distance matrix for the positives: ~{len(pos_idx) ** 2 * 4 / 1e9:.2f} GB")

10,221 rows: 221 positives, 10,000 negatives; ECFP4 has 2048 bits
Distance matrix for the positives: ~0.00 GB


## 3. Cluster the positives

In [5]:
%%time
from sklearn.cluster import AgglomerativeClustering

# Tanimoto distance between binary fingerprints (int32 counts, so large fingerprints cannot overflow)
X = X_pos.astype(np.int32)
intersection = X @ X.T
bit_counts = X.sum(axis=1, keepdims=True)
union = bit_counts + bit_counts.T - intersection
dist = 1.0 - np.divide(intersection, union, out=np.zeros(intersection.shape, dtype=np.float32),
                       where=union != 0).astype(np.float32)
np.fill_diagonal(dist, 0.0)
del X, intersection, union

cluster_labels = AgglomerativeClustering(
    n_clusters=N_CLUSTERS, metric="precomputed", linkage=LINKAGE
).fit_predict(dist)
del dist

cluster_to_pos = {cid: pos_idx[cluster_labels == cid] for cid in np.unique(cluster_labels)}
sizes = sorted((len(m) for m in cluster_to_pos.values()), reverse=True)
print(f"{len(cluster_to_pos)} clusters of positives, sizes: {sizes}")

20 clusters of positives, sizes: [131, 20, 17, 12, 7, 6, 5, 3, 3, 3, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1]
CPU times: user 2.8 s, sys: 204 ms, total: 3.01 s
Wall time: 1.27 s


## 4. Build the folds

Test positives come from whole clusters, and every positive in those clusters is left out of that fold's
training set. Test negatives are sampled fresh for each fold.

In [6]:
rng = np.random.default_rng(SEED)
unused = set(cluster_to_pos)
test_pos_groups, excluded_pos_groups, fold_clusters = [], [], []

while len(test_pos_groups) < TARGET_N_FOLDS:
    available = sorted(((c, cluster_to_pos[c]) for c in unused), key=lambda x: len(x[1]), reverse=True)
    if not available or sum(len(m) for _, m in available) < TARGET_TEST_POS_PER_FOLD:
        break
    # Start from the largest unused cluster; add the smallest others until there are enough positives
    seed_cid, seed_members = available[0]
    selected, members = [seed_cid], list(seed_members)
    if len(members) < TARGET_TEST_POS_PER_FOLD:
        for cid, m in sorted(((c, cluster_to_pos[c]) for c in unused if c != seed_cid), key=lambda x: len(x[1])):
            if len(members) >= TARGET_TEST_POS_PER_FOLD:
                break
            selected.append(cid)
            members.extend(m)
    if len(members) < TARGET_TEST_POS_PER_FOLD:
        break
    members = np.array(members)
    test_pos_groups.append(np.sort(rng.choice(members, size=TARGET_TEST_POS_PER_FOLD, replace=False)))
    excluded_pos_groups.append(np.sort(members))
    fold_clusters.append(selected)
    unused -= set(selected)

assert test_pos_groups, "No folds could be built from the clusters; try fewer TARGET_TEST_POS_PER_FOLD"

rng = np.random.default_rng(SEED)  # the pipeline uses a fresh RNG with the same seed for the negatives
n_test_neg = int(round(len(neg_idx) * TEST_NEG_FRACTION))
folds = []
for test_pos, excluded_pos in zip(test_pos_groups, excluded_pos_groups):
    test_neg = rng.choice(neg_idx, size=n_test_neg, replace=False)
    test_idx = np.sort(np.concatenate([test_pos, test_neg]))
    train_mask = np.ones(len(y), dtype=bool)
    train_mask[test_neg] = False
    train_mask[excluded_pos] = False
    folds.append((np.where(train_mask)[0], test_idx))

N_FOLDS = len(folds)
pd.DataFrame([
    dict(fold=i, clusters=str(fold_clusters[i]), train=len(tr), test=len(te),
         train_pos=int(y[tr].sum()), test_pos=int(y[te].sum()),
         held_out_pos=len(excluded_pos_groups[i]) - int(y[te].sum()))
    for i, (tr, te) in enumerate(folds)
])

,fold,clusters,train,test,train_pos,test_pos,held_out_pos
0,0,[2],7590,2600,90,100,31


## 5. Save the CSV and fold indices

`held_out_pos` above counts positives from the test clusters that are in neither the fold's training nor
its test set. The CSV keeps the rows in the same order as the indices; `parquet_row` is each row's position
in the original parquet file.

In [7]:
stem = TRAIN_PATH.stem
csv_path = OUTPUT_DIR / f"{stem}_cluster_cv_data.csv"
npz_path = OUTPUT_DIR / f"{stem}_cluster_cv_folds.npz"  # fixed name; the fold count is inside

df[[SMILES_COL, LABEL_COL, "parquet_row"]].to_csv(  # same column names as the parquet
    csv_path, index=False
)
np.savez(npz_path, **{f"fold{i}_{part}": idx for i, (tr, te) in enumerate(folds)
                      for part, idx in (("train", tr), ("test", te))})

print(f"Saved {csv_path}  ({len(df):,} rows)")
print(f"Saved {npz_path}  ({N_FOLDS} folds)")
print("\nIn the k-fold notebooks, set:")
print(f'  DATA_PATH = Path("{csv_path}")')
print(f'  CV_INDICES_PATH = Path("{npz_path}")')

Saved ../output/sample-data/cv_folds/train_cluster_cv_data.csv  (10,221 rows)
Saved ../output/sample-data/cv_folds/train_cluster_cv_folds.npz  (1 folds)

In the k-fold notebooks, set:
  DATA_PATH = Path("../output/sample-data/cv_folds/train_cluster_cv_data.csv")
  CV_INDICES_PATH = Path("../output/sample-data/cv_folds/train_cluster_cv_folds.npz")
